# Stage 1 v2 — Initial ISBI pre-training
This rerun uses the stable spatial-softmax/KL/smooth-L1 objective and selects the primary checkpoint by validation loss. Set `ISBI_ROOT` below to the actual ISBI dataset location. The repository discovery and data-loading workflow are preserved.


In [ ]:
import sys
from pathlib import Path

# Locate the repository before importing the shared .py modules. If src/ is
# stored elsewhere, replace SRC_DIR with that absolute directory.
REPO_ROOT = Path('/kaggle/input/models/pouyansajadi78/model-sep-17-v1/tensorflow2/default/1')
for start in (Path.cwd().resolve(), Path('/kaggle/working')):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'config.py').is_file():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is not None:
        break
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate repository root containing src/config.py')
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# The dataset can be outside the repository. Config is frozen, so provide the
# path to the factory instead of trying to assign cfg.dataset_root later.
ISBI_ROOT = '/kaggle/input/datasets/pouyansajadi78/cropped-isbi-dataset/cropped_isbi_dataset/ISBI Dataset'
from config import make_stage1_initial_config
cfg = make_stage1_initial_config(dataset_root=ISBI_ROOT)
print('Repository root:', REPO_ROOT)
print('Source directory:', SRC_DIR)
print('ISBI dataset root:', cfg.dataset_root)
print(cfg)
assert cfg.decoder_filters == (1024, 512, 256)
assert cfg.epochs == 150 and cfg.warmup_epochs == 5
assert cfg.objective_version == 'v2'
assert cfg.decoding_version == 'spatial_softmax'
assert cfg.heatmap_loss == 'spatial_kl' and cfg.heatmap_loss_weight == 0.1
assert cfg.coord_loss == 'smooth_l1' and cfg.coord_loss_weight == 10.0
assert cfg.checkpoint_monitor == 'val_loss'
assert cfg.output_filename == 'best_model_fixed.weights.h5'
assert cfg.best_mre_output_filename == 'best_model_stage1_v2_best_mre.weights.h5'

In [ ]:
from data import ISBIDataset, CephalometricDataLoader
from augmentation import CephalometricAugmentation
from model import build_model
from train import train
train_dataset = ISBIDataset(cfg.dataset_root, cfg, mode='train')
val_dataset = ISBIDataset(cfg.dataset_root, cfg, mode='test')
train_loader = CephalometricDataLoader(train_dataset, cfg, shuffle=True, augmentation=CephalometricAugmentation(cfg))
val_loader = CephalometricDataLoader(val_dataset, cfg, shuffle=False, augmentation=None)
model, encoder_layer_names = build_model(cfg)
model.summary()

In [ ]:
# Keep online training augmented, but evaluate the same training cohort without
# augmentation so its curve is directly comparable with validation.
train_eval_dataset = ISBIDataset(cfg.dataset_root, cfg, mode='train')
train_eval_loader = CephalometricDataLoader(
    train_eval_dataset,
    cfg,
    shuffle=False,
    augmentation=None,
)

history = train(
    cfg,
    train_loader=train_loader,
    train_eval_loader=train_eval_loader,
    val_loader=val_loader,
    model=model,
    encoder_layer_names=encoder_layer_names,
)
history_path = Path(history['history_path'])
print('History keys:', list(history))
print('Saved history:', history_path)
print('Primary best-loss checkpoint:', history['checkpoint_paths']['primary'])
print('Secondary best-MRE checkpoint:', history['checkpoint_paths']['best_mre'])


In [ ]:
import matplotlib.pyplot as plt

epochs = history['epoch']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].plot(epochs, history['train_eval_loss'], label='Train (unaugmented eval)')
axes[0, 0].plot(epochs, history['val_loss'], label='Validation (raw)')
axes[0, 0].plot(epochs, history['val_loss_running_best'], '--', label='Validation running best')
axes[0, 0].set(title='Stage 1 v2 total loss', xlabel='Epoch', ylabel='Loss')
axes[0, 0].legend()

axes[0, 1].plot(epochs, history['val_weighted_hm_loss'], label='0.1 × spatial KL')
axes[0, 1].plot(epochs, history['val_weighted_coord_loss'], label='10 × smooth-L1')
axes[0, 1].set(title='Validation loss components', xlabel='Epoch', ylabel='Weighted loss')
axes[0, 1].legend()

axes[1, 0].plot(epochs, history['train_eval_mre'], label='Train (unaugmented eval)')
axes[1, 0].plot(epochs, history['val_mre'], label='Validation')
axes[1, 0].set(title='Mean radial error', xlabel='Epoch', ylabel='MRE (mm)')
axes[1, 0].legend()

axes[1, 1].plot(epochs, history['learning_rate'])
axes[1, 1].set(title='Learning rate', xlabel='Epoch', ylabel='Learning rate')
axes[1, 1].set_yscale('log')

for axis in axes.flat:
    axis.grid(alpha=0.25)
fig.tight_layout()
plt.show()


In [ ]:
restored = history.get('restored_final_validation') or {}
print(f"Best validation loss: {history['best_loss_value']:.6f} at epoch {history['best_loss_epoch']}")
print(f"Best validation MRE:  {history['best_mre_value']:.3f} mm at epoch {history['best_mre_epoch']}")
print(f"Stopped at epoch {history['stopped_epoch']}: {history['stop_reason']}")
print('Restored checkpoint:', history.get('restored_checkpoint'))
if restored:
    print(f"Restored validation loss: {restored['loss']:.6f}")
    print(f"Restored validation MRE:  {restored['mre']:.3f} mm")
